# RLSSM Quickstart: Instantiation, Model Building, and Sampling

This notebook provides a minimal end-to-end demonstration of the `RLSSM` class:

1. **Load** a balanced-panel two-armed bandit dataset
2. **Define** an annotated learning function and the angle SSM log-likelihood
3. **Configure** and **instantiate** an `RLSSM` model
4. **Inspect** the built Bambi / PyMC model
5. **Run** a minimal 2-draw sampling smoke test

For a full treatment — simulating data, hierarchical formulas, meaningful sampling, and posterior visualization — see:
- [rlssm_tutorial.ipynb](rlssm_tutorial.ipynb)
- [add_custom_rlssm_model.ipynb](add_custom_rlssm_model.ipynb)

## 1. Imports and Setup

In [1]:
from pathlib import Path

import jax.numpy as jnp
import numpy as np
import pandas as pd

import hssm
from hssm.rl import RLSSM, RLSSMConfig
from hssm.distribution_utils.onnx import make_jax_matrix_logp_funcs_from_onnx
from hssm.rl.likelihoods.two_armed_bandit import compute_v_subject_wise
from hssm.utils import annotate_function

# RLSSM requires float32 throughout (JAX default).
hssm.set_floatX("float32", update_jax=True)

Setting PyTensor floatX type to float32.


Setting "jax_enable_x64" to False. If this is not intended, please set `jax` to False.


## 2. Load the Dataset

We use a small synthetic two-armed bandit dataset from the HSSM test fixtures.  
It is a **balanced panel**: every participant has the same number of trials.  
Columns: `participant_id`, `trial_id`, `rt`, `response`, `feedback`.

> **Note:** You can also generate data with
> [`ssm-simulators`](https://github.com/AlexanderFengler/ssm-simulators).
> See `rlssm_tutorial.ipynb` for an example.

In [2]:
# Path relative to docs/tutorials/ when running inside the HSSM repo.
_fixture_path = Path("../../tests/fixtures/rldm_data.npy")
raw = np.load(_fixture_path, allow_pickle=True).item()
data = pd.DataFrame(raw["data"])

n_participants = data["participant_id"].nunique()
n_trials = len(data) // n_participants

print(data.head())
print(f"\nParticipants: {n_participants}  |  Trials per participant: {n_trials}")

   participant_id  trial  response        rt  feedback  correct
0               0      0       0.0  0.935602  0.126686      0.0
1               0      1       0.0  1.114379  0.173100      0.0
2               0      2       0.0  0.564311  0.444935      0.0
3               0      3       0.0  2.885860  0.307207      0.0
4               0      4       0.0  0.532113  0.177911      0.0

Participants: 20  |  Trials per participant: 200


## 3. Define the Learning Process

The RL learning process is a JAX function that, given a subject's trial sequence, computes
the trial-wise drift rate `v` via a Q-learning update rule.  

`annotate_function` attaches `.inputs`, `.outputs`, and (optionally) `.computed` metadata
that the RLSSM likelihood builder uses to automatically construct the input matrix for the
decision process.

- **inputs** — columns that the function reads (free parameters + data columns)
- **outputs** — what the function produces (here: `v`, the drift rate)

Here we annotate the built-in `compute_v_subject_wise` function, which implements a simple
Rescorla-Wagner Q-learning update for a two-armed bandit task.

In [3]:
compute_v_annotated = annotate_function(
    inputs=["rl_alpha", "scaler", "response", "feedback"],
    outputs=["v"],
)(compute_v_subject_wise)

print("Learning function inputs :", compute_v_annotated.inputs)
print("Learning function outputs:", compute_v_annotated.outputs)

Learning function inputs : ['rl_alpha', 'scaler', 'response', 'feedback']
Learning function outputs: ['v']


## 4. Define the Decision (SSM) Log-Likelihood

The decision process uses the **angle model** likelihood, loaded from an ONNX file.
`make_jax_matrix_logp_funcs_from_onnx` returns a JAX callable that accepts a
2-D matrix whose columns are `[v, a, z, t, theta, rt, response]` and returns
per-trial log-probabilities.

We then annotate that callable so the builder knows:
- which columns the matrix contains (`inputs`)
- that `v` itself is *computed* by the learning function (not a free parameter)

The ONNX file is loaded from the local test fixture when running inside the HSSM
repository; otherwise it is downloaded from the HuggingFace Hub (`franklab/HSSM`).

In [4]:
# Use the local fixture when available; fall back to HuggingFace download.
_local_onnx = Path("../../tests/fixtures/angle.onnx").resolve()
_onnx_model = str(_local_onnx) if _local_onnx.exists() else "angle.onnx"

_angle_logp_jax = make_jax_matrix_logp_funcs_from_onnx(model=_onnx_model)

angle_logp_func = annotate_function(
    inputs=["v", "a", "z", "t", "theta", "rt", "response"],
    outputs=["logp"],
    computed={"v": compute_v_annotated},
)(_angle_logp_jax)

print("SSM logp inputs :", angle_logp_func.inputs)
print("SSM logp outputs:", angle_logp_func.outputs)
print("Computed deps   :", list(angle_logp_func.computed.keys()))

SSM logp inputs : ['v', 'a', 'z', 't', 'theta', 'rt', 'response']
SSM logp outputs: ['logp']
Computed deps   : ['v']


## 5. Configure the Model with `RLSSMConfig`

`RLSSMConfig` collects all the information the RLSSM class needs:

| Field | Purpose |
|-------|---------|
| `model_name` | Identifier string for the configuration |
| `decision_process` | Name of the SSM (e.g. `"angle"`) |
| `list_params` | Ordered list of *free* parameters to sample |
| `params_default` | Starting / default values for each parameter |
| `bounds` | Prior bounds for each parameter |
| `learning_process` | Dict mapping computed param name → annotated learning function |
| `extra_fields` | Extra data columns required by the learning function |
| `ssm_logp_func` | Annotated JAX callable for the decision-process likelihood |

In [5]:
rlssm_config = RLSSMConfig(
    model_name="rlssm_angle_quickstart",
    loglik_kind="approx_differentiable",
    decision_process="angle",
    decision_process_loglik_kind="approx_differentiable",
    learning_process_kind="blackbox",
    list_params=["rl_alpha", "scaler", "a", "theta", "t", "z"],
    params_default=[0.1, 1.0, 1.0, 0.0, 0.3, 0.5],
    bounds={
        "rl_alpha": (0.0, 1.0),
        "scaler": (0.0, 10.0),
        "a": (0.1, 3.0),
        "theta": (-0.1, 0.1),
        "t": (0.001, 1.0),
        "z": (0.1, 0.9),
    },
    learning_process={"v": compute_v_annotated},
    response=["rt", "response"],
    choices=[0, 1],
    extra_fields=["feedback"],
    ssm_logp_func=angle_logp_func,
)

print("Model name  :", rlssm_config.model_name)
print("Free params :", rlssm_config.list_params)

Model name  : rlssm_angle_quickstart
Free params : ['rl_alpha', 'scaler', 'a', 'theta', 't', 'z']


## 6. Instantiate the `RLSSM` Model

Passing `data` and `rlssm_config` to `RLSSM`:

- validates the balanced-panel requirement
- builds a differentiable PyTensor Op that chains the RL learning step and the
  angle log-likelihood
- constructs the Bambi / PyMC model internally

Note that `v` (the drift rate) is *not* a free parameter — it is computed inside
the Op by the Q-learning update and therefore does not appear in `model.params`.

In [6]:
model = RLSSM(data=data, model_config=rlssm_config)

assert isinstance(model, RLSSM)
print("Model type       :", type(model).__name__)
print("Participants     :", model.n_participants)
print("Trials/subj      :", model.n_trials)
print("Free parameters  :", list(model.params.keys()))
assert "rl_alpha" in model.params, "rl_alpha must be a free parameter"
assert "v" not in model.params, "v is computed, not a free parameter"
model

You supplied a model 'rlssm_angle_quickstart', which is currently not supported in the ssm_simulators package. An error will be thrown when sampling from the random variable or when using any posterior or prior predictive sampling methods.


Model initialized successfully.


Model type       : RLSSM
Participants     : 20
Trials/subj      : 200
Free parameters  : ['rl_alpha', 'scaler', 'a', 'theta', 't', 'z', 'p_outlier']


Hierarchical Sequential Sampling Model
Model: rlssm_angle_quickstart

Response variable: rt,response
Likelihood: approx_differentiable
Observations: 4000

Parameters:

rl_alpha:
    Prior: Uniform(lower: 0.0, upper: 1.0)
    Explicit bounds: (0.0, 1.0)

scaler:
    Prior: Uniform(lower: 0.0, upper: 10.0)
    Explicit bounds: (0.0, 10.0)

a:
    Prior: Uniform(lower: 0.10000000149011612, upper: 3.0)
    Explicit bounds: (0.1, 3.0)

theta:
    Prior: Uniform(lower: -0.10000000149011612, upper: 0.10000000149011612)
    Explicit bounds: (-0.1, 0.1)

t:
    Prior: Uniform(lower: 0.0010000000474974513, upper: 1.0)
    Explicit bounds: (0.001, 1.0)

z:
    Prior: Uniform(lower: 0.10000000149011612, upper: 0.8999999761581421)
    Explicit bounds: (0.1, 0.9)


Lapse probability: 0.05
Lapse distribution: Uniform(lower: 0.0, upper: 20.0)

## 7. Inspect the Built Model

After construction, `model.model` exposes the underlying **Bambi model** and
`model.pymc_model` exposes the **PyMC model** context — useful for debugging
or customizing priors.

In [7]:
print("=== Bambi model ===")
print(model.model)

print("\n=== PyMC model ===")
print(model.pymc_model)

=== Bambi model ===
       Formula: c(rt, response) ~ 1
        Family: SSM Family
          Link: rl_alpha = identity
  Observations: 4000
        Priors: 
    target = rl_alpha
        Common-level effects
            Intercept ~ Uniform(lower: 0.0, upper: 1.0)
        
        Auxiliary parameters
            scaler ~ Uniform(lower: 0.0, upper: 10.0)
            p_outlier ~ 0.05
            a ~ Uniform(lower: 0.10000000149011612, upper: 3.0)
            z ~ Uniform(lower: 0.10000000149011612, upper: 0.8999999761581421)
            theta ~ Uniform(lower: -0.10000000149011612, upper: 0.10000000149011612)
            t ~ Uniform(lower: 0.0010000000474974513, upper: 1.0)

=== PyMC model ===


## 8. Sampling

A minimal sampling run — 2 draws, 2 tuning steps, 1 chain — confirms that the full
computational graph (Q-learning scan → angle logp → NUTS gradient) is wired correctly.

In [8]:
trace = model.sample(draws=2, tune=2, chains=1, cores=1, sampler="numpyro", target_accept=0.9)

assert trace is not None
print(trace)

Using default initvals. 



Only 2 samples per chain. Reliable r-hat and ESS diagnostics require longer chains for accurate estimate.


/Users/yxu150/HSSM/.venv/lib/python3.13/site-packages/pytensor/gradient.py:1327: FutureWarning: LANLogpOp should implement `pullback` instead of `L_op`/`grad`. Direct `L_op`/`grad` implementations are deprecated and will stop being called in a future version.
  input_grads = node.op.pullback(inputs, node.outputs, new_output_grads)


/Users/yxu150/HSSM/.venv/lib/python3.13/site-packages/bambi/backend/pymc.py:224: UserWarning: `init='adapt_diag'` is ignored by `nuts_sampler='numpyro'`; the external sampler uses its own initialization.
  idata = pm.sample(
NUTS[numpyro]: [scaler, a, z, theta, t, rl_alpha]


/Users/yxu150/HSSM/.venv/lib/python3.13/site-packages/jax/_src/numpy/array_methods.py:125: UserWarning: Explicitly requested dtype float64 requested in astype is not available, and will be truncated to dtype float32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  return lax_numpy.astype(self, dtype, copy=copy, device=device)


...

sample: 100%|██████████| 4/4 [00:00<00:00,  5.94it/s, 1 steps of size 4.13e-01. acc. prob=0.00]


There were 2 divergences after tuning. Increase `target_accept` or reparameterize.


The number of samples is too small to check convergence reliably.


/Users/yxu150/HSSM/.venv/lib/python3.13/site-packages/pytensor/link/numba/dispatch/basic.py:214: UserWarning: Numba will use object mode to run LANLogpOp's perform method. Set `pytensor.config.compiler_verbose = True` to see more details.
  warnings.warn(


<xarray.DataTree>
Group: /
├── Group: /posterior
│       Dimensions:   (chain: 1, draw: 2)
│       Coordinates:
│         * chain     (chain) int64 8B 0
│         * draw      (draw) int64 16B 0 1
│       Data variables:
│           scaler    (chain, draw) float32 8B ...
│           a         (chain, draw) float32 8B ...
│           theta     (chain, draw) float32 8B ...
│           t         (chain, draw) float32 8B ...
│           z         (chain, draw) float32 8B ...
│           rl_alpha  (chain, draw) float32 8B ...
│       Attributes:
│           created_at:                  2026-06-29T18:40:19.070366+00:00
│           creation_library:            ArviZ
│           creation_library_version:    1.1.0
│           creation_library_language:   Python
│           sample_dims:                 ['chain', 'draw']
│           inference_library:           numpyro
│           inference_library_version:   0.21.0
│           sampling_time:               2.351171
│           tuning_steps:       

## Summary

This notebook showed how to:

1. Load a balanced-panel dataset (`rldm_data.npy`)
2. Annotate a Q-learning function with `annotate_function`
3. Load the angle ONNX likelihood and annotate it so the builder can assemble the input matrix
4. Define an `RLSSMConfig` and pass it to `RLSSM`
5. Confirm model structure (free params, Bambi / PyMC objects)
6. Run a 2-draw sampling smoke test that returns an `arviz.InferenceData` object